# Prompt chaining workflow with Pydantic AI

Prompt chaining splits a complex task into multiple subtasks. Each subtask is handled by a dedicated agent, and the output of one feeds into the next. This gives better results but at the cost of higher latency.

```mermaid
flowchart LR
    In --> LLM1["LLM Call 1"]
    LLM1 -- "Output 1" --> Gate{Gate}
    Gate -- Pass --> LLM2["LLM Call 2"]
    Gate -- Fail --> Exit[Exit]
    LLM2 -- "Output 2" --> LLM3["LLM Call 3"]
    LLM3 --> Out
```

**Examples:**
- Generating content in a pipeline: table of contents, content, revisions, translations
- Multi-step text processing with quality gates between steps

In [ ]:
import nest_asyncio

nest_asyncio.apply()

## Setup

In [ ]:
import logfire
from dotenv import load_dotenv
from pydantic_ai import Agent

load_dotenv()

logfire.configure()
logfire.instrument_pydantic_ai()

## Vanilla workflow

We create three agents, each handling one step of the pipeline:

1. Generate a table of contents
2. Generate the article content
3. Revise the content if it's too long

In [ ]:
toc_agent = Agent(
    "openai:gpt-5-nano",
    system_prompt=(
        "You are an expert writer specialized in SEO. Provided with a topic, "
        "you will generate the table of contents for a short article."
    ),
)

article_agent = Agent(
    "openai:gpt-5-nano",
    system_prompt=(
        "You are an expert writer specialized in SEO. Provided with a topic and a table of contents, "
        "you will generate the content of the article."
    ),
)

editor_agent = Agent(
    "openai:gpt-5-nano",
    system_prompt=(
        "You are an expert writer specialized in SEO. Provided with a topic, a table of contents and content, "
        "you will revise the content of the article to make it less than 1000 characters."
    ),
)


def run_workflow(topic: str) -> str:
    toc = toc_agent.run_sync(
        f"Generate the table of contents of an article about {topic}"
    )
    content = article_agent.run_sync(
        f"Generate the content of an article about {topic} with the following table of contents: {toc.output}"
    )
    if len(content.output) > 1000:
        revised = editor_agent.run_sync(
            f"Revise the content of an article about {topic} with the following table of contents: {toc.output} "
            f"and the following content: {content.output}"
        )
        return revised.output
    return content.output


output = run_workflow("Artificial Intelligence")
print(output)

# Exercise

Build a workflow to generate recipes. If the recipe includes more than 5 ingredients, revise the recipe to try to exclude some ingredients if possible.

You should be able to compare the initial and revised recipe and ingredients.

In [ ]:
from pydantic import BaseModel, Field


class Ingredients(BaseModel):
    ingredients: list[str] = Field(description="Ingredients required for the recipe")


class RevisedRecipe(BaseModel):
    ingredients: list[str]
    recipe_content: str


ingredient_agent = Agent(
    "openai:gpt-5-nano",
    output_type=Ingredients,
    system_prompt=(
        "You are an expert chef. Given a recipe request, return only the ingredients."
    ),
)

recipe_agent = Agent(
    "openai:gpt-5-nano",
    system_prompt=(
        "You are an expert chef. Given a recipe request and ingredients, "
        "write a clear recipe."
    ),
)

recipe_editor = Agent(
    "openai:gpt-5-nano",
    output_type=RevisedRecipe,
    system_prompt=(
        "You are an expert chef. Given a recipe request, ingredients, "
        "and a recipe, remove unnecessary ingredients when possible. "
        "Aim for 5 ingredients or fewer."
    ),
)


def run_recipe_workflow(recipe_request: str) -> dict:
    ingredients_result = ingredient_agent.run_sync(
        f"List the ingredients for {recipe_request}."
    )
    ingredients = ingredients_result.output.ingredients

    recipe_result = recipe_agent.run_sync(
        f"Write a recipe for {recipe_request} using these ingredients: {ingredients}"
    )

    output = {
        "ingredients": ingredients,
        "recipe_content": recipe_result.output,
        "revised_ingredients": ingredients,
        "revised_recipe_content": recipe_result.output,
    }

    if len(ingredients) > 5:
        revised = recipe_editor.run_sync(
            f"Recipe request: {recipe_request}\n\n"
            f"Ingredients: {ingredients}\n\n"
            f"Recipe:\n{recipe_result.output}"
        ).output
        output["revised_ingredients"] = revised.ingredients
        output["revised_recipe_content"] = revised.recipe_content

    return output

In [ ]:
recipe = run_recipe_workflow("a healthy breakfast")

print("Initial ingredients:", recipe["ingredients"])
print(recipe["recipe_content"])
print("\nRevised ingredients:", recipe["revised_ingredients"])
print(recipe["revised_recipe_content"])